#### Set Up LLAMA3.1 LLM

In [1]:
#Importing packages
from typing import Dict, TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain.output_parsers import CommaSeparatedListOutputParser
import os

In [2]:
from langchain_community.chat_models import ChatOllama

model = ChatOllama(
        model = "llama3.1",                  # Llama3.1 Model
        temperature = 0.8,                   # Control creativity (0.0 - basic | 1.0 - creative)
        num_predict = 256                    # Limit the number of tokens generated
)

/tmp/ipykernel_5030/3237006782.py:3: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  model = ChatOllama(


In [3]:
# Define a function to interact with the LLM
def llm(prompt: str) -> str:
    response = model.predict(prompt)         # Call the Llama model
    return response                          # Extract the response content

#### Required StateGrapgh Variables

In [4]:
#Setting StateGraph variables

class GraphState(TypedDict):
    all_history: Optional[str]=None
    history: Optional[str] = None
    result: Optional[str]=None
    final_result: Optional[str]=None
    total_questions: Optional[int]=None
    interviewer: Optional[str]=None
    candidate: Optional[str]=None
    current_question: Optional[str]=None
    current_answer: Optional[str]=None
    round: Optional[int] = None
    total_rounds: Optional[int] =None
    total_questions_per_round: Optional[int] = None

#### Creating a Workflow

In [5]:
workflow = StateGraph(GraphState)

#### Define the prompt set required alongside the nodes for the graph object.

In [6]:
#Different Agent prompts used

prompt_interviewer = "You're a {}. You need to interview a {}. This is the interview so far:\n{}\n\
Ask your next question and dont repeat your questions.\
Output just the question and no extra text"

prompt_result = "Evaluate the performance of the candidate based on the response given for the questions asked\
    give a rating on a scale of 100. Explain the rating in very short paragraph.\nThe interview:\n{}"

prompt_verdict="Given the interview, should we select the candidate?\
    Give output as Yes or No with a reason.\nThe interview:\n{}"

#Defining nodes for LangGraph
def handle_question(state):
    # Retrieve current history, role, candidate, and last answer
    history = state.get('history', '').strip()
    role = state.get('interviewer', '').strip()
    candidate = state.get('candidate', '').strip()
    last_answer = state.get('current_answer', '').strip()
    last_question = state.get('current_question', '').strip()

    if last_answer:
        # Chain Step 1: Summarize the candidate's response
        summarize_prompt = (
            f"The candidate's previous answer was: '{last_answer}'\n"
            "Summarize this answer in one or two sentences, focusing on key points for further discussion."
        )
        summary = llm(summarize_prompt)

        # Chain Step 2: Generate a follow-up question
        follow_up_prompt = (
            f"As an interviewer ({role}), you are conducting a detailed interview with {candidate}.\n"
            f"The previous question was: '{last_question}'\n"
            f"The summarized response was: '{summary}'\n"
            "Based on this response, ask a thoughtful follow-up question to extend the discussion. Output only the question and no other statement."
        )
        question = role + ": " + llm(follow_up_prompt)
    else:
        # For the first question, use the original job description prompt
        prompt = prompt_interviewer.format(role, candidate, history)
        question = role + ": " + llm(prompt)

    # Reset history if it's empty or 'Nothing'
    if history == 'Nothing':
        history = ''

    # Update the state with the new question
    return {
        "history": history + '\n\n' + question,
        "current_question": question,
        "total_questions": state.get('total_questions', 0) + 1,
    }

# Handle user input (your response)
def handle_response(state):
    history = state.get('history', '').strip()
    question = state.get('current_question', '').strip()
    candidate = state.get('candidate', '').strip()

    # Manually input the answer in the terminal
    print(f"Question: {question}")

    # Take input from the user as the answer
    answer = input("Your Answer: ")                            

    # Append the answer to the history
    return {
        "history":history+'\n\n'+answer,
        "current_answer":answer
    }

# Handle the feedback for the interview
def handle_result(state):
    history = state.get('history', '').strip()
    interviewer = state.get('interviewer', '').strip()

    # Evaluate the interview performance and provide feedback using Ollama's llama3.1 model
    prompt = prompt_result.format(history)
    result = llm(prompt)
    all_history = state.get('all_history', '').strip()
    
    result = f"Interviewed by {interviewer}\n{result}"

    return {
        "result": state.get('result', '') + '\n' + result,
        "history": 'Nothing',
        "all_history": all_history + '\n\nInterviewed by ' + interviewer + '\n' + history,
        'final_result': result
    }

# Handle the final selection
def handle_selection(state):
    result = state.get('result', '').strip()
    prompt = prompt_verdict.format(result)
    verdict = llm(prompt)
    
    return{"final_result":verdict}

#### Add the above defined nodes into the workflow grapgh object

In [7]:
workflow.add_node("handle_question",handle_question)
workflow.add_node("handle_response",handle_response)
workflow.add_node("handle_result",handle_result)
workflow.add_node("handle_selection",handle_selection)

#### Add required conditional edges

In [8]:
#Defining conditional edges

def check_conv_length(state):
    if state.get("total_questions", 0) < 5:
        return "handle_question"
    else:
        return "handle_result"

def check_rounds(state):
    if state.get("total_questions", 0) >= 5 and state.get("total_rounds", 1) == 1:
        return "handle_selection"
    else:
        return "handle_question"

#### Adding Conditional edges to the workflow

In [9]:
# Add conditional edges for handle_response(to workflow)
if "handle_response" not in workflow.branches:
    workflow.add_conditional_edges(
        "handle_response", 
        check_conv_length,  
        {
            "handle_question": "handle_question",  
            "handle_result": "handle_result"     
        }
    )

# Add conditional edges for handle_result ( to workflow)
if "handle_result" not in workflow.branches:
    workflow.add_conditional_edges(
        "handle_result",  
        check_rounds,     
        {
            "handle_question": "handle_question",  
            "handle_selection": "handle_selection"  
        }
    )

# Set the entry point for the workflow
workflow.set_entry_point("handle_question")
workflow.add_edge('handle_question', "handle_response")
workflow.add_edge('handle_selection',END)

#### Job Description

In [10]:
job_description = input("Enter the job description: ")

#### Compile the grapgh object

In [11]:
app = workflow.compile()

#### Invoke the grapgh object and initiate interview

In [12]:
# Start the interview process with AI as the interviewer
conversation = app.invoke({
    'total_questions': 0,  
    'candidate':job_description ,  
    'total_rounds': 1,  
})

# Output the final result after the interview
final_feedback = conversation.get("final_result")
print(f"Final Feedback: {final_feedback}")
print(conversation['result'])
print(conversation['all_history'])

/tmp/ipykernel_5030/2342275772.py:3: LangChainDeprecationWarning: The method `BaseChatModel.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = model.predict(prompt)         # Call the Llama model


Question: : How do you optimize the performance of a Python script when dealing with large datasets?
Question: : How did you approach learning about optimizing performance in Python when you first encountered large datasets, and can you walk me through your process?


KeyboardInterrupt: 